In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# ==============================
# Base paths
# ==============================
BASE = r'C:\Users\Anushka Thite\Desktop\smart-loan-system'
PROCESSED_DIR = os.path.join(BASE, 'data', 'processed')
MODELS_DIR = os.path.join(BASE, 'models')

# ==============================
# Load preprocessed data
# ==============================
X_train = np.load(os.path.join(PROCESSED_DIR, 'X_train.npy'))
X_test = np.load(os.path.join(PROCESSED_DIR, 'X_test.npy'))
y_train = np.load(os.path.join(PROCESSED_DIR, 'y_train.npy'))
y_test = np.load(os.path.join(PROCESSED_DIR, 'y_test.npy'))

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")
print(f"Train target balance: {np.bincount(y_train)}")
print(f"Test target balance: {np.bincount(y_test)}\n")

# ==============================
# Define models
# ==============================
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=42, eval_metric='logloss', n_jobs=-1
    ),
    'SVM': SVC(probability=True, kernel='rbf', C=1.0, random_state=42)
}

# ==============================
# Train and evaluate
# ==============================
results = {}
trained_models = {}

for name, model in models.items():
    print(f"{'=' * 55}")
    print(f"Training: {name}")
    print(f"{'=' * 55}")

    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results[name] = {
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'AUC': round(auc, 4),
        'Train_Time_sec': round(train_time, 2)
    }
    trained_models[name] = model

    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1       : {f1:.4f}")
    print(f"  AUC      : {auc:.4f}")
    print(f"  Time     : {train_time:.2f} sec")
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Rejected', 'Approved']))
    print()

    fname = name.replace(" ", "_").lower() + '.pkl'
    joblib.dump(model, os.path.join(MODELS_DIR, fname))

# ==============================
# Comparison table
# ==============================
print("\n" + "=" * 75)
print("MODEL COMPARISON SUMMARY")
print("=" * 75)
results_df = pd.DataFrame(results).T
print(results_df.to_string())

results_df.to_csv(os.path.join(MODELS_DIR, 'baseline_results.csv'))
print(f"\n✅ Saved results to models/baseline_results.csv")
print(f"✅ Saved trained models to models/*.pkl")

X_train: (20570, 63), X_test: (4000, 63)
y_train: (20570,), y_test: (4000,)
Train target balance: [10285 10285]
Test target balance: [1429 2571]

Training: Logistic Regression
  Accuracy : 0.9487
  Precision: 0.9621
  Recall   : 0.9580
  F1       : 0.9600
  AUC      : 0.9897
  Time     : 0.54 sec

  Classification Report:
              precision    recall  f1-score   support

    Rejected       0.93      0.93      0.93      1429
    Approved       0.96      0.96      0.96      2571

    accuracy                           0.95      4000
   macro avg       0.94      0.95      0.94      4000
weighted avg       0.95      0.95      0.95      4000


Training: KNN
  Accuracy : 0.9100
  Precision: 0.9434
  Recall   : 0.9148
  F1       : 0.9289
  AUC      : 0.9628
  Time     : 0.02 sec

  Classification Report:
              precision    recall  f1-score   support

    Rejected       0.85      0.90      0.88      1429
    Approved       0.94      0.91      0.93      2571

    accuracy          

C:\Users\Anushka Thite\Desktop\smart-loan-system\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


  Accuracy : 0.9473
  Precision: 0.9542
  Recall   : 0.9642
  F1       : 0.9592
  AUC      : 0.9876
  Time     : 143.89 sec

  Classification Report:
              precision    recall  f1-score   support

    Rejected       0.93      0.92      0.93      1429
    Approved       0.95      0.96      0.96      2571

    accuracy                           0.95      4000
   macro avg       0.94      0.94      0.94      4000
weighted avg       0.95      0.95      0.95      4000



MODEL COMPARISON SUMMARY
                     Accuracy  Precision  Recall      F1     AUC  Train_Time_sec
Logistic Regression    0.9487     0.9621  0.9580  0.9600  0.9897            0.54
KNN                    0.9100     0.9434  0.9148  0.9289  0.9628            0.02
Random Forest          0.9457     0.9482  0.9685  0.9582  0.9899            2.42
XGBoost                0.9527     0.9529  0.9747  0.9637  0.9919            1.69
SVM                    0.9473     0.9542  0.9642  0.9592  0.9876          143.89

✅ Saved r

In [2]:
import pandas as pd
results_df = pd.DataFrame(results).T
print(results_df.to_string())

                     Accuracy  Precision  Recall      F1     AUC  Train_Time_sec
Logistic Regression    0.9487     0.9621  0.9580  0.9600  0.9897            0.54
KNN                    0.9100     0.9434  0.9148  0.9289  0.9628            0.02
Random Forest          0.9457     0.9482  0.9685  0.9582  0.9899            2.42
XGBoost                0.9527     0.9529  0.9747  0.9637  0.9919            1.69
SVM                    0.9473     0.9542  0.9642  0.9592  0.9876          143.89
